In [1]:
from typing import Callable

import pandas as pd
import polars as pl
from pathlib import Path
from datetime import datetime, date

def getBatch(inputFunction : Callable) -> Callable:
    def innerWrap(inSeries : pl.Series):
        return pl.Series(map(inputFunction, inSeries))
    return innerWrap

@getBatch
def stripString(x: str | None) -> str | None:
    if x:
        return x.strip()
    return None


@getBatch
def parseDate(x: str) -> date:
    return datetime.strptime(x,"%m-%d-%Y").date()


@getBatch
def parseFloat(x:str) -> float:
    return float(x.replace(",",'')) 

@getBatch
def int2String(x:int | None) -> str | None:
    if x:
        return f"{x}"
    return None

@getBatch
def zeroPadded7(x:str | None) -> str | None:
    if x:
    # temp= 7 - len(x)
        return f"{x:0>7}"
    return None

loc_cols = ['GFPipeID', 'PipeName', 'GFLocID', 'GF_LocID_Tag', 'LocID', 'Loc', 'GF_LocName', 'LocName', 'GF_FacilityType', 'GF_FacilityTypeGroup', 'ReportedFacilityType', 'LocSegment', 'LocZone', 'State', 'County', 'InterconnectingEntity', 'UpdatedTime']

fact_cols =['GasMonth','GasDay','Dataset','GFLocID','LocName','DesignCapacity','OperatingCapacity','TotalScheduledQuantity','OperationallyAvailableCapacity','IT','FlowDirection','Timestamp',]

raw_NN_path = Path('.').absolute().parent / 'downloads/enbridge/NN_raw'

pipe_configs_path = Path('.').absolute().parent / 'src/artifacts/configFiles/PipeConfigs.parquet'

pipesDf = pl.read_parquet(pipe_configs_path)

metaDf =pl.read_csv(pipe_configs_path.parent / "metaLocs.csv")


In [2]:
temp_list =[]
for i in raw_NN_path.iterdir():
    pipe_code = i.name.split('_',1)[0].strip()
    temp = pd.read_csv(i,dtype=str)
    if not temp.empty:
        temp_list.append(temp.assign(PipeCode=pipe_code))
    
df_NN : pd.DataFrame= pd.concat(temp_list,ignore_index=True)
df = pl.from_dataframe(df=df_NN.astype(str))

In [11]:
df["Direction_Of_Flow"].unique()

Direction_Of_Flow
str
"""R"""
"""B"""
"""D"""


In [3]:
NN_cols_selected = ['Effective_From_DateTime',
#  'Effective_End_DateTime',
 'Loc_Prop',
 'Loc_Name',
 'Allocated_Qty',
 'Direction_Of_Flow',
#  'Accounting_Physincal_Indicator',
#  'Posting_DateTime',
 'PipeCode']

NN_cols = ['Effective_From_DateTime',
 'Effective_End_DateTime',
 'Loc_Prop',
 'Loc_Name',
 'Allocated_Qty',
 'Direction_Of_Flow',
 'Accounting_Physincal_Indicator',
 'Posting_DateTime',
 'PipeCode']

# fact_cols =['GasMonth','GasDay','Dataset','GFLocID','LocName','DesignCapacity','OperatingCapacity','TotalScheduledQuantity','OperationallyAvailableCapacity','IT','FlowDirection','Timestamp',]


NN_gold_Map ={
    'Effective_From_DateTime': 'GasMonth',
#  'Effective_End_DateTime',
 'Loc_Prop':'Loc',
 'Loc_Name': 'LocName',
 'Allocated_Qty':'',
 'Direction_Of_Flow':'FlowDirection',
#  'Accounting_Physincal_Indicator',
#  'Posting_DateTime',
 'PipeCode': 'PipeCode'
}

meta_map = {
    'Loc St Abbrev':'State',
    'Loc Cnty': 'County',
    'Loc Zone':'LocZone',
    'Loc Type Ind':'ReportedFacilityType',
    }

meta_cols_selected = [
    'Loc St Abbrev',
    'Loc Cnty',
    'Loc Zone',
    'Loc Type Ind',
    "GFLocID"
]

In [4]:
NNLocs=df["Loc_Prop", "Loc_Name","PipeCode"].unique().join(pipesDf["GFPipeID","PipeName","PipeCode"], on="PipeCode",how="left")\
.with_columns(
    pl.col("Loc_Prop").str.strip_chars().alias("Loc"),
    pl.col("Loc_Name").alias("LocName"),
    pl.col("Loc_Prop").str.strip_chars().str.zfill(7).alias("LocID"),


    pl.concat_str(
        [
            pl.col("GFPipeID"),
            pl.col("Loc_Prop").str.strip_chars().str.zfill(7)

        ]
    ).alias("GFLocID"),
    pl.lit(datetime.now()).cast(pl.Datetime).alias("UpdatedTime"),


).join(metaDf[meta_cols_selected],on="GFLocID", how="left").rename(meta_map).rename({})


In [9]:
NNLocs.with_columns(
    *[pl.lit(None).cast(pl.String).alias(col_name) for col_name in loc_cols if col_name not in NNLocs.columns],
).select(loc_cols).fill_null("")

# .write_csv(pipe_configs_path.parent / "NNLocMeta.csv")


GFPipeID,PipeName,GFLocID,GF_LocID_Tag,LocID,Loc,GF_LocName,LocName,GF_FacilityType,GF_FacilityTypeGroup,ReportedFacilityType,LocSegment,LocZone,State,County,InterconnectingEntity,UpdatedTime
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,datetime[μs]
"""100""","""Algonquin Gas Transmission, LL…","""1000000815""","""""","""0000815""","""00815""","""""","""Dighton Power LLC Bristol MA (…","""""","""""","""END""","""""","""""","""MA""","""BRISTOL""","""""",2026-03-23 09:41:40.296409
"""120""","""Texas Eastern Transmission, LP""","""1200070238""","""""","""0070238""","""70238""","""""","""Hughes Eastern - Chickasaw, MS""","""""","""""","""WHD""","""""","""1""","""MS""","""CHICKASAW""","""""",2026-03-23 09:41:40.296409
"""100""","""Algonquin Gas Transmission, LL…","""1000002210""","""""","""0002210""","""02210""","""""","""LAMBERTVILLE DELIVERY (HUNTERD…","""""","""""","""INT""","""""","""""","""NJ""","""HUNTERDON""","""""",2026-03-23 09:41:40.296409
"""120""","""Texas Eastern Transmission, LP""","""1200073715""","""""","""0073715""","""73715""","""""","""Lancaster Municipal Gas Compan…","""""","""""","""LDC""","""""","""2""","""OH""","""FAIRFIELD""","""""",2026-03-23 09:41:40.296409
"""120""","""Texas Eastern Transmission, LP""","""1200070104""","""""","""0070104""","""70104""","""""","""Lafayette, TN""","""""","""""","""LDC""","""""","""1""","""TN""","""MACON""","""""",2026-03-23 09:41:40.296409
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""120""","""Texas Eastern Transmission, LP""","""1200070098""","""""","""0070098""","""70098""","""""","""Fulton, MS""","""""","""""","""LDC""","""""","""1""","""MS""","""ITAWAMBA""","""""",2026-03-23 09:41:40.296409
"""120""","""Texas Eastern Transmission, LP""","""1200073455""","""""","""0073455""","""73455""","""""","""McBee Operating Company, LLC-R…","""""","""""","""WHD""","""""","""ETX""","""TX""","""NACOGDOCHES""","""""",2026-03-23 09:41:40.296409
"""120""","""Texas Eastern Transmission, LP""","""1200071402""","""""","""0071402""","""71402""","""""","""Creal Springs, IL""","""""","""""","""LDC""","""""","""1""","""IL""","""WILLIAMSON""","""""",2026-03-23 09:41:40.296409


In [6]:
df.columns

['Effective_From_DateTime',
 'Effective_End_DateTime',
 'Loc_Prop',
 'Loc_Name',
 'Allocated_Qty',
 'Direction_Of_Flow',
 'Accounting_Physincal_Indicator',
 'Posting_DateTime',
 'PipeCode']

In [7]:
pipesDf.collect_schema()

Schema([('GFPipeID', String),
        ('MetaCode', String),
        ('NoNoticeCode', String),
        ('ParentPipe', String),
        ('PipeCode', String),
        ('PipeName', String),
        ('PointCapCode', String),
        ('SegmentCapCode', String),
        ('StorageCapCode', Null)])

In [8]:
LocsDf = df.select(OA_cols_gold).rename(mapping=OA_gold_map).select(["Loc","LocName","PipeCode"]).join(pipesDf["PipeCode","GFPipeID","PipeName"], on="PipeCode").unique()

LocsDf=LocsDf.with_columns(
    pl.col("Loc").str.strip_chars(),
    pl.col("Loc").str.strip_chars().str.zfill(7).alias("LocID")
    # pl.col("Loc").str.strip_chars().map_batches(zeroPadded7,return_dtype=pl.String).alias("LocID"),
).with_columns(
    pl.concat_str([
        pl.col("GFPipeID"),
        pl.col("LocID")
    ]).alias("GFLocID"),
)



# LocsDf.write_csv(pipe_configs_path.parent / "OALocMeta.csv")

NameError: name 'OA_cols_gold' is not defined

In [ ]:
LocsDf

In [ ]:
metaDf =pl.read_csv(pipe_configs_path.parent / "metaLocs.csv")

In [ ]:
meta_map = {
    'Loc St Abbrev':'State',
    'Loc Cnty': 'County',
    'Loc Zone':'LocZone',
    'Loc Type Ind':'ReportedFacilityType',
    }

meta_cols_selected = [
    'Loc St Abbrev',
    'Loc Cnty',
    'Loc Zone',
    'Loc Type Ind',
    "GFLocID"
]

In [ ]:
loc_cols

In [ ]:
LocsDf =LocsDf.join(metaDf[meta_cols_selected],on="GFLocID", how="left").rename(mapping=meta_map)

In [ ]:
LocsDf.with_columns(
    *[pl.lit(None).cast(pl.String).alias(col_name) for col_name in loc_cols if col_name not in LocsDf.columns]
).select(loc_cols).fill_null("")
# .write_csv(pipe_configs_path.parent / "OALocMeta.csv")

In [ ]:
silverOA = df.join(pipesDf["PipeCode","GFPipeID"], on="PipeCode", how="left").with_columns(
    pl.concat_str(
        [
            pl.col("GFPipeID"),
            pl.col("Loc").str.zfill(7)

        ]
    ).alias("GFLocID")
).select(pl.exclude(["PipeCode","GFPipeID"])).unique()

In [ ]:
goldOADf=df.select(OA_cols_gold).rename(OA_gold_map).join(pipesDf["GFPipeID","PipeCode"],on="PipeCode",how="left")\
.with_columns(

    pl.col("GasDay").str.to_date(format="%m-%d-%Y"),
    
    pl.lit(datetime.now()).cast(pl.Datetime).alias("Timestamp"),

    pl.lit("OA").alias("Dataset"),

    pl.col("Loc").str.strip_chars(),

    pl.col("DesignCapacity").str.replace_all(",",""),
    pl.col("OperatingCapacity").str.replace_all(",",""),
    pl.col("TotalScheduledQuantity").str.replace_all(",",""),
    pl.col("OperationallyAvailableCapacity").str.replace_all(",",""),

)\
.with_columns(
    pl.col("Loc").str.zfill(7).alias("LocID"),

    pl.col("DesignCapacity").str.to_decimal(scale=2),
    pl.col("OperatingCapacity").str.to_decimal(scale=2),
    pl.col("TotalScheduledQuantity").str.to_decimal(scale=2),
    pl.col("OperationallyAvailableCapacity").str.to_decimal(scale=2),
)\
.with_columns(
    pl.concat_str([
        pl.col("GFPipeID"),
        pl.col("LocID")
    ]).alias("GFLocID"),
    
    pl.concat_str(
        [
            pl.col("GasDay").dt.year().cast(pl.String),
            pl.col("GasDay").dt.month().cast(pl.String).str.zfill(2)
        ]
    ).str.to_integer().alias("GasMonth")
).select(fact_cols)

goldOADf
